# Dataset Preprocessing Pipeline

## Libraries

In [1]:
import xarray as xr # Climate data
import pandas as pd # CSV and dataframes
import geopandas as gpd # Geographical data and spatial joins
import numpy as np 

from os import listdir
from pyproj import Transformer # Coordinates projection

## Data Loading, Bounding And Cleaning

#### Loading Of Fires Data

In [2]:
df = pd.read_excel("data/fire_data.xlsx")

starting_rows = len(df)

numeric_cols = ['Huso', 'CoordenadaX', 'CoordenadaY']

other_cols = ['Detectado', 'Datum']

for c in numeric_cols:
    df[c] = df[c].astype(str).str.replace(',', '.') # In spanish statistical data sometimes decimals use the comma format
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['Detectado'] = pd.to_datetime(df['Detectado'], errors='coerce', format='%d/%m/%Y %H:%M:%S').dt.normalize()


df.dropna(subset=['CoordenadaX', 'CoordenadaY'], inplace=True) # Drop fires without coordinates
df.drop(columns = [c for c in df.columns if c not in numeric_cols + other_cols], inplace=True) # Only use the needed columns

df = df[df["Huso"].between(29.0, 31.0)] # Drop some detected handwriting errors


The coordinates are in two different formats (EPSG:23030 for the old data and EPSG:25830 for the new data). Both formats are converted into EPSG:4326. Some rows have manual errors, a wide range is applied to clean outlier values.

In [3]:
old_29_transformer = Transformer.from_crs("EPSG:23029", "EPSG:4326", always_xy=True)
old_30_transformer = Transformer.from_crs("EPSG:23030", "EPSG:4326", always_xy=True)
old_31_transformer = Transformer.from_crs("EPSG:23031", "EPSG:4326", always_xy=True)
new_29_transformer = Transformer.from_crs("EPSG:25829", "EPSG:4326", always_xy=True)
new_30_transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
new_31_transformer = Transformer.from_crs("EPSG:25831", "EPSG:4326", always_xy=True)

# All the needed coordinate converters
transformers = {
    (29, False): old_29_transformer,
    (30, False): old_30_transformer,
    (31, False): old_31_transformer,
    (29, True):  new_29_transformer,
    (30, True):  new_30_transformer,
    (31, True):  new_31_transformer,
}

mask_etrs = df['Datum'].astype(str).str.contains('ETRS89', case=False, na=False) # Mask to distinct between old and new formats

df['Longitude'] = float('nan')
df['Latitude'] = float('nan')

# Converts old coordinates to new unified coordinates
for (huso, is_etrs), group in df.groupby(['Huso', mask_etrs]):
    transformer = transformers[(int(huso), bool(is_etrs))]
    lon, lat = transformer.transform(
        group['CoordenadaX'].values,
        group['CoordenadaY'].values
    )
    df.loc[group.index, 'Longitude'] = lon
    df.loc[group.index, 'Latitude'] = lat


df = df[df['Longitude'].between(-8, -1.5) & df['Latitude'].between(39.5, 43.5)] # Ensure the fires are inside the desired zone

df.drop(columns=['Huso', 'CoordenadaX', 'CoordenadaY', 'Datum'], inplace=True) # Drop auxiliary columns, keep only date and coordinates

print(f"Removed rows: {starting_rows - len(df)}")
print(f"Remaining rows: {len(df)}")

Removed rows: 1825
Remaining rows: 27537


#### Definition Of Borders And Time Limits

In [4]:
first_fire = df['Detectado'].min()
last_fire = df['Detectado'].max()

min_lat, max_lat = df['Latitude'].min(), df['Latitude'].max()
min_lon, max_lon = df['Longitude'].min(), df['Longitude'].max()

print(f"First fire: {first_fire}")
print(f"Last fire: {last_fire}")
print(f"Latitude range: {min_lat} to {max_lat}")
print(f"Longitude range: {min_lon} to {max_lon}")

start_time = first_fire - pd.Timedelta(days=90) # First fire - rolling window
end_time = last_fire

First fire: 2006-01-21 00:00:00
Last fire: 2023-12-30 00:00:00
Latitude range: 40.090201004204765 to 43.23304854759588
Longitude range: -7.065729152572756 to -1.8077660689453114


#### Loading, Cleaning And Bounding Of The Climate Data

In [5]:
elevation_ds = xr.open_dataset("data/elev_ens_0.1deg_reg_v33.0e.nc")
wind_speed_ds = xr.open_dataset("data/fg_ens_mean_0.1deg_reg_v33.0e.nc")
rel_humidity_ds = xr.open_dataset("data/hu_ens_mean_0.1deg_reg_v33.0e.nc")
preasure_sl_ds = xr.open_dataset("data/pp_ens_mean_0.1deg_reg_v33.0e.nc")
radiation_ds = xr.open_dataset("data/qq_ens_mean_0.1deg_reg_v33.0e.nc")
rain_ds = xr.open_dataset("data/rr_ens_mean_0.1deg_reg_v33.0e.nc")
avg_temperature_ds = xr.open_dataset("data/tg_ens_mean_0.1deg_reg_v33.0e.nc")
min_temperature_ds = xr.open_dataset("data/tn_ens_mean_0.1deg_reg_v33.0e.nc")
max_temperature_ds = xr.open_dataset("data/tx_ens_mean_0.1deg_reg_v33.0e.nc")


elevation_ds = elevation_ds.sel(longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
wind_speed_ds = wind_speed_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
rel_humidity_ds = rel_humidity_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
preasure_sl_ds = preasure_sl_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
radiation_ds = radiation_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
rain_ds = rain_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
avg_temperature_ds = avg_temperature_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
min_temperature_ds = min_temperature_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))
max_temperature_ds = max_temperature_ds.sel(time=slice(start_time, end_time), longitude=slice(min_lon, max_lon), latitude=slice(min_lat, max_lat))

#### Loading Of Demographic Data

In [6]:
provinces_df = []

for f in listdir("./data/provincias"):
    provinces_df.append(pd.read_csv('./data/provincias/' + f, sep=";", encoding="latin-1", thousands=".")) # Thousands="." is needed because of the data format

dem_df = pd.concat(provinces_df, ignore_index=True).drop(columns=["Sexo"])

dem_df = pd.DataFrame({
    "INE": dem_df["Municipios"].str[:5],
    "Year": dem_df["Periodo"].astype(int),
    "Population": pd.to_numeric(dem_df["Total"].astype(str).str.replace(".", "", regex=False), errors="coerce"),
})

#### Loading Of Terrain And Vegetation Data

In [7]:
mfe = gpd.read_file("data/mfe_cyl/MFE_41.shp") # Keep in mind this is a geodataframe not a normal one


# Names can be a little bit confusing (look at the dictionary I'll be providing in the docs)
numeric_vars_mfe = [
    "FCCHER", "FCCTOT", "HMMAT",
    "O1", "O2", "O3",
    "FCCARB", "FCCMAT"
]

categorical_vars_mfe = [
    "TipesTR25", "Especie1", "Especie2",
    "Especie3", "Estado1", "Estado2",
    "Estado3", "Distribuci", "TIPOBOSQUE",
    "n_MODCOM", "n_REGBIO"
]


mfe = mfe[numeric_vars_mfe + categorical_vars_mfe + ["geometry"]] # Drop auxiliary columns

The categorical variables are mapped by a risk score. By doing this, the model has access to better information while lowering the splits needed.

In [8]:
mapa_riesgo_especies = {
    "sin datos": 0,
    "Salix spp.": 1, "Salix atrocinerea": 1, "Salix alba": 1, "Salix fragilis": 1,
    "Salix caprea": 1, "Salix elaeagnos": 1, "Salix purpurea": 1, "Salix babylonica": 1,
    "Alnus glutinosa": 1, "Populus x canadensis": 1, "Populus nigra": 1, "Populus alba": 1,
    "Populus tremula": 1, "Fraxinus angustifolia": 1, "Fraxinus excelsior": 1, "Fraxinus ornus": 1,
    "Ulmus minor": 1, "Ulmus glabra": 1, "Ulmus pumila": 1, "Tamarix spp.": 1,
    "Sambucus nigra": 1, "Platanus hispanica": 1,
    "Fagus sylvatica": 2, "Ficus carica": 2, "Quercus pyrenaica": 2, "Quercus faginea": 2,
    "Quercus petraea": 2, "Quercus humilis (Q. Pubescens)": 2, "Quercus canariensis": 2,
    "Quercus robur": 2, "Quercus rubra": 2, "Castanea sativa": 2, "Betula alba": 2,
    "Betula spp.": 2, "Betula pendula": 2, "Acer campestre": 2, "Acer monspessulanum": 2,
    "Acer pseudoplatanus": 2, "Acer negundo": 2, "Acer opalus": 2, "Acer spp.": 2,
    "Sorbus aucuparia": 2, "Sorbus aria": 2, "Sorbus spp.": 2, "Sorbus torminalis": 2,
    "Prunus spp.": 2, "Prunus avium": 2, "Prunus padus": 2, "Corylus avellana": 2,
    "Juglans regia": 2, "Juglans nigra": 2, "Crataegus monogyna": 2, "Crataegus spp.": 2,
    "Celtis australis": 2, "Tilia spp.": 2, "Tilia platyphyllos": 2, "Tilia cordata": 2,
    "Pyrus spp.": 2, "Malus sylvestris": 2, "Robinia pseudoacacia": 2, "Ailanthus altissima": 2,
    "Carpinus betulus": 2, "Gleditsia triacanthos": 2, "Mezcla de pequeñas frondosas": 2,
    "Mezcla de frondosas de gran porte": 2, "Otras frondosas": 2,
    "Quercus ilex": 3, "Quercus suber": 3, "Prunus lusitanica": 3, "Ilex aquifolium": 3,
    "Juniperus thurifera": 3, "Juniperus communis": 3, "Juniperus oxycedrus": 3,
    "Juniperus phoenicea": 3, "Arbutus unedo": 3, "Phillyrea latifolia": 3,
    "Olea europaea": 3, "Taxus baccata": 3,
    "Pinus pinea": 4, "Pinus nigra": 4, "Pinus sylvestris": 4, "Pinus uncinata": 4,
    "Otra Coniferas": 4, "Mezcla de coníferas": 4, "Cupressus arizonica": 4,
    "Cupressus sempervirens": 4, "Cupressus macrocarpa": 4, "Pseudotsuga menziesii": 4,
    "Picea abies": 4, "Abies alba": 4, "Abies pinsapo": 4, "Thuja spp": 4,
    "Cedrus atlantica": 4, "Cedrus deodara": 4, "Chamaecyparis lawsoniana": 4,
    "Larix decidua": 4, "Larix spp.": 4, "Larix x eurolepis": 4,
    "Eucalyptus camaldulensis": 5, "Eucalyptus globulus": 5, "Eucalyptus nitens": 5,
    "Pinus halepensis": 5, "Pinus pinaster": 5, "Pinus radiata": 5,
    "Pinus canariensis": 5, "Acacia spp.": 5
}

mapa_estado_ordinal = {
    "Sin datos": 0,
    "Fustal": 1,
    "Repoblado": 2,
    "Latizal": 3,
    "Monte bravo": 4
}

mapa_distribucion = {
    "No procede": 0,
    "Piés aislados": 1,
    "Discontinua en bosquetes": 2,
    "Discontinua en fajas": 3,
    "Discontinua en mosaico": 4,
    "Discontinua irregular": 5,
    "Uniforme": 6
}

r1 = mfe["Especie1"].map(mapa_riesgo_especies).fillna(0)
r2 = mfe["Especie2"].map(mapa_riesgo_especies).fillna(0)
r3 = mfe["Especie3"].map(mapa_riesgo_especies).fillna(0)

o1 = mfe["O1"].fillna(0)
o2 = mfe["O2"].fillna(0)
o3 = mfe["O3"].fillna(0)

total_ocupacion = o1 + o2 + o3

mfe["Riesgo_Combustible"] = np.where(
    total_ocupacion > 0,
    (r1 * o1 + r2 * o2 + r3 * o3) / total_ocupacion,
    0
)

mfe["Estado1_num"] = mfe["Estado1"].map(mapa_estado_ordinal).fillna(0)
mfe["Distribuci_num"] = mfe["Distribuci"].map(mapa_distribucion).fillna(0)

mfe = pd.get_dummies(mfe, columns=["TIPOBOSQUE"], prefix="tipo_bosque", dtype=int)

mfe.drop(
    columns=[
        "Especie1", "Especie2", "Especie3", 
        "O1", "O2", "O3", 
        "Estado1", "Estado2", "Estado3", 
        "Distribuci"
    ],
    inplace=True,
    errors="ignore"
)

mfe = mfe.to_crs("EPSG:4326")

## Feature Extraction

In [9]:
ds_list = [
    rain_ds,
    rel_humidity_ds,
    max_temperature_ds,
    min_temperature_ds,
    avg_temperature_ds,
    radiation_ds,
    wind_speed_ds,
    preasure_sl_ds,
]

# float64 -> float32 to avoid RAM overload
# Also .load() is used to avoid using dask which can increase consumption
ds_list = [ds.astype(np.float32).load() for ds in ds_list]

(
    rain_ds,
    rel_humidity_ds,
    max_temperature_ds,
    min_temperature_ds,
    avg_temperature_ds,
    radiation_ds,
    wind_speed_ds,
    preasure_sl_ds,
) = ds_list


# Rolling statistics
rain_ds["rr_7d_sum"] = rain_ds["rr"].rolling(time=7, min_periods=1).sum()
rain_ds["rr_30d_sum"] = rain_ds["rr"].rolling(time=30, min_periods=1).sum()
rain_ds["rr_90d_sum"] = rain_ds["rr"].rolling(time=90, min_periods=1).sum()

rel_humidity_ds["hu_7d_mean"] = (
    rel_humidity_ds["hu"].rolling(time=7, min_periods=1).mean()
)
rel_humidity_ds["hu_30d_mean"] = (
    rel_humidity_ds["hu"].rolling(time=30, min_periods=1).mean()
)

max_temperature_ds["tx_3d_max"] = (
    max_temperature_ds["tx"].rolling(time=3, min_periods=1).max()
)
max_temperature_ds["tx_7d_mean"] = (
    max_temperature_ds["tx"].rolling(time=7, min_periods=1).mean()
)

min_temperature_ds["tn_3d_min"] = (
    min_temperature_ds["tn"].rolling(time=3, min_periods=1).min()
)
min_temperature_ds["tn_7d_mean"] = (
    min_temperature_ds["tn"].rolling(time=7, min_periods=1).mean()
)
min_temperature_ds["tn_14d_mean"] = (
    min_temperature_ds["tn"].rolling(time=14, min_periods=1).mean()
)
min_temperature_ds["tn_30d_mean"] = (
    min_temperature_ds["tn"].rolling(time=30, min_periods=1).mean()
)

radiation_ds["qq_7d_sum"] = (
    radiation_ds["qq"].rolling(time=7, min_periods=1).sum()
)
radiation_ds["qq_30d_sum"] = (
    radiation_ds["qq"].rolling(time=30, min_periods=1).sum()
)

wind_speed_ds["fg_3d_max"] = (
    wind_speed_ds["fg"].rolling(time=3, min_periods=1).max()
)
wind_speed_ds["fg_30d_max"] = (
    wind_speed_ds["fg"].rolling(time=30, min_periods=1).max()
)

# Elevation derivatives
elevation_ds = elevation_ds.load()
dz_dx = elevation_ds["elevation"].differentiate("longitude")
dz_dy = elevation_ds["elevation"].differentiate("latitude")
elevation_ds["slope"] = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
elevation_ds["aspect_south"] = np.cos(np.arctan2(-dz_dy, dz_dx))

# Merge
climate_ds = xr.merge(
    [
        rain_ds,
        rel_humidity_ds,
        max_temperature_ds,
        min_temperature_ds,
        avg_temperature_ds,
        radiation_ds,
        wind_speed_ds,
        preasure_sl_ds,
    ],
    join="override",
)

# Climate derivatives
climate_ds["wind_dryness_ratio"] = climate_ds["fg"] / (
    climate_ds["hu"] + 1e-5
)
climate_ds["fanning_index"] = (
    climate_ds["fg"] * (100 - climate_ds["hu"]) * climate_ds["tx"]
)
climate_ds["moisture_deficit_30d"] = (
    climate_ds["qq_30d_sum"] * climate_ds["tx_7d_mean"]
) / (climate_ds["rr_30d_sum"] + 1)

is_dry = (climate_ds["rr"] < 1.0).astype(np.int32)
climate_ds["cdd"] = (
    is_dry.cumsum(dim="time")
    - is_dry.cumsum(dim="time").where(is_dry == 0).ffill(dim="time").fillna(0)
)

# Day of year encoding
day = climate_ds["time"].dt.dayofyear
climate_ds["day_sin"] = np.sin(2 * np.pi * day / 365.25).astype(np.float32)
climate_ds["day_cos"] = np.cos(2 * np.pi * day / 365.25).astype(np.float32)

## Negative Dates And Position Sampling
---
Two strategies are combined, each with its own negative to positive ratio:
- Repeating dates
- Random uniform distribution

In [10]:
REPEAT_NEGATIVES_RATIO = 1 # Same date as positives
RANDOM_NEGATIVES_RATIO = 1 # Uniform distribution

dates_neg_repeat = df['Detectado'].repeat(REPEAT_NEGATIVES_RATIO).reset_index(drop=True) # Repeated dates


rand_ns = np.random.uniform(first_fire.value, end_time.value, size=len(df) * RANDOM_NEGATIVES_RATIO).astype('int64') # Random dates (start_time + rolling window, so every negative has full rolling data)
dates_neg_random = pd.Series(pd.to_datetime(rand_ns).normalize())


target_dates_neg = pd.concat([dates_neg_repeat, dates_neg_random], ignore_index=True)

valid_chunks = []
total_needed = len(target_dates_neg)
current_count = 0

# For each date, some random coordinates are generated. If they fall in an appropriate terrain, they're saved.
while current_count < total_needed:
    left = total_needed - current_count

    rand_lons = np.random.uniform(min_lon, max_lon, size=left)
    rand_lats = np.random.uniform(min_lat, max_lat, size=left)
    
    batch_gdf = gpd.GeoDataFrame(
        geometry=gpd.points_from_xy(rand_lons, rand_lats),
        crs="EPSG:4326"
    )
    
    in_forest = gpd.sjoin(batch_gdf, mfe, how="inner", predicate="within")
    
    if not in_forest.empty:
        valid_chunks.append(in_forest)
        current_count += len(in_forest)

valid_negatives = pd.concat(valid_chunks).iloc[:total_needed].copy()

valid_negatives['Detectado'] = target_dates_neg.values
valid_negatives['Longitude'] = valid_negatives.geometry.x
valid_negatives['Latitude'] = valid_negatives.geometry.y
valid_negatives['Label'] = 0

negatives_df = pd.DataFrame(valid_negatives[['Detectado', 'Longitude', 'Latitude', 'Label']])

df["Label"] = 1
df = pd.concat([df, negatives_df], ignore_index=True) # Join positive and negative dates

df = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]), crs="EPSG:4326").sort_values(by='Detectado').reset_index(drop=True)

## Data Merging 

In [11]:
munis = gpd.read_file("./data/municipios/recintos_municipales_inspire_peninbal_etrs89.shp")
munis["INE"] = munis["NATCODE"].str[-5:] # The INE code corresponds to [-5:] of "NATCODE" column
munis = munis[["INE", "geometry"]].to_crs(df.crs) # Drop unneeded columns

df = gpd.sjoin(df, munis, how="left", predicate="within").drop(columns="index_right")

df["Year"] = df["Detectado"].dt.year
df = df.merge(dem_df, on=["INE", "Year"], how="left", validate="m:1").drop(columns="Year") # Merges both datasets

df.loc[df["INE"].str.startswith("53", na=False), "Population"] = 0 # This INE code usually means no population

print(f"Points without municipality: {df['INE'].isna().sum()}")
print(f"Points without population:   {df['Population'].isna().sum()}")

df.drop(columns=["INE"], inplace = True) # Drop auxiliary columns

Points without municipality: 24
Points without population:   74


In [12]:
df.dropna(inplace=True)

# Join df with terrain + vegetation data 
full_df = gpd.sjoin(df, mfe, how="left", predicate="within").drop(
    columns=["index_right", "geometry"]
)

lon = xr.DataArray(full_df["Longitude"].values, dims="points")
lat = xr.DataArray(full_df["Latitude"].values, dims="points")
time = xr.DataArray(full_df["Detectado"].values, dims="points")

# Selection of data in desired points
sampled_climate = climate_ds.sel(
    longitude=lon, latitude=lat, time=time, method="nearest"
)

# Join with climate data
for var in climate_ds.data_vars:
    full_df[var] = sampled_climate[var].values


sampled_elev = elevation_ds[["elevation", "slope", "aspect_south"]].sel(
    longitude=lon, latitude=lat, method="nearest"
)

# Join with elevation data
for var in sampled_elev.data_vars:
    full_df[var] = sampled_elev[var].values

In [13]:
# Final column names for readability
column_names = {
    "Detectado": "date",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Label": "fire",
    "Population": "population",
    "FCCHER": "herbaceous_cover",
    "FCCTOT": "total_cover",
    "HMMAT": "shrub_height",
    "FCCARB": "tree_cover",
    "FCCMAT": "shrub_cover",
    "TipesTR25": "structure_type",
    "n_MODCOM": "fuel_model",
    "n_REGBIO": "biogeographic_region",
    "Riesgo_Combustible": "species_fuel_risk",
    "Estado1_num": "development_stage",
    "Distribuci_num": "stand_distribution",
    "tipo_bosque_-": "forest_type_none",
    "tipo_bosque_Coníferas": "forest_type_conifer",
    "tipo_bosque_Frondosas": "forest_type_broadleaf",
    "tipo_bosque_Masas mixtas": "forest_type_mixed",
    "rr": "precipitation",
    "rr_7d_sum": "precipitation_7d_sum",
    "rr_30d_sum": "precipitation_30d_sum",
    "rr_90d_sum": "precipitation_90d_sum",
    "hu": "humidity",
    "hu_7d_mean": "humidity_7d_mean",
    "hu_30d_mean": "humidity_30d_mean",
    "tx": "max_temperature",
    "tx_3d_max": "max_temperature_3d_max",
    "tx_7d_mean": "max_temperature_7d_mean",
    "tn": "min_temperature",
    "tn_3d_min": "min_temperature_3d_min",
    "tn_7d_mean": "min_temperature_7d_mean",
    "tn_14d_mean": "min_temperature_14d_mean",
    "tn_30d_mean": "min_temperature_30d_mean",
    "tg": "mean_temperature",
    "qq": "radiation",
    "qq_7d_sum": "radiation_7d_sum",
    "qq_30d_sum": "radiation_30d_sum",
    "fg": "wind_speed",
    "fg_3d_max": "wind_speed_3d_max",
    "fg_30d_max": "wind_speed_30d_max",
    "pp": "sea_level_pressure",
    "cdd": "consecutive_dry_days",
    "doy_sin": "day_of_year_sin",
    "doy_cos": "day_of_year_cos",
}

full_df.rename(columns=column_names, inplace=True)

full_df.to_csv("./dataset.csv", index=False)